In [ ]:
from pathlib import Path

import duckdb
import matplotlib.pyplot as plt
from matplotlib.ticker import StrMethodFormatter

# read_only so the notebook can never modify the database, and never takes the write lock
# that a `duckdb` CLI session in another terminal would then be refused.
con = duckdb.connect("../data/processed/grid.duckdb", read_only=True)
con.execute("SET TimeZone = 'UTC'")
# Runs the Q4 file in this connection, so its TEMP tables, scenario_day among them, exist here.
con.execute(open("../analysis/04_shifting_gain.sql").read())


def q(sql):
    """Run SQL and return a pandas DataFrame, so matplotlib has a table to draw."""
    return con.execute(sql).df()


# EUR saved per MW per year for 1 to 12 hours moved a day, per block size: the last statement
# of analysis/04_shifting_gain.sql, unrounded. Expected rows: 24.
df = q("""
    SELECT block, k, 365 * AVG(saving) AS eur_per_mw_year
    FROM scenario_day
    GROUP BY block, k
    ORDER BY block, k
""")
hour = df[df["block"] == "hour"]
quarter = df[df["block"] == "quarter"]

# The first two slots of the reference categorical palette, in fixed order; text stays in ink.
BLUE, ORANGE, INK, MUTED = "#2a78d6", "#eb6834", "#0b0b0b", "#52514e"

fig, ax = plt.subplots(figsize=(8, 4.5))
# Quarter-hours plotted first so the legend lists the lines in the order they sit.
ax.plot(
    quarter["k"],
    quarter["eur_per_mw_year"],
    color=ORANGE,
    linewidth=2,
    marker="o",
    markersize=6,
    label="Moved in quarter-hours",
)
ax.plot(
    hour["k"],
    hour["eur_per_mw_year"],
    color=BLUE,
    linewidth=2,
    marker="o",
    markersize=6,
    label="Moved in hours",
)

# Direct labels at the end of each line, and the two values the memo quotes for hours.
for series, text, dy in ((quarter, "in quarter-hours", 6), (hour, "in hours", -12)):
    last = series.iloc[-1]
    ax.annotate(
        text,
        (last["k"], last["eur_per_mw_year"]),
        xytext=(8, dy),
        textcoords="offset points",
        color=MUTED,
        fontsize=9,
    )
for k in (1, 4):
    row = hour[hour["k"] == k].iloc[0]
    ax.annotate(
        f"€{round(row['eur_per_mw_year'], -2):,.0f}",
        (k, row["eur_per_mw_year"]),
        xytext=(4, -16),
        textcoords="offset points",
        color=INK,
        fontsize=9,
    )

ax.set_xticks(range(1, 13))
ax.set_xlim(0.5, 13.8)
ax.set_ylim(0, None)
ax.yaxis.set_major_formatter(StrMethodFormatter("€{x:,.0f}"))
ax.set_xlabel("Hours moved out of the dearest part of each day", color=MUTED)
ax.set_ylabel("Saved per MW per year", color=MUTED)
ax.tick_params(colors=MUTED)
ax.spines[["top", "right"]].set_visible(False)
ax.spines[["left", "bottom"]].set_color("#c9c8c4")
ax.grid(axis="y", color="#e6e5e1", linewidth=0.8)
ax.set_axisbelow(True)
ax.legend(frameon=False, loc="upper left")
ax.set_title(
    "The first hours moved each day are worth the most", loc="left", fontsize=13, color=INK, pad=12
)
fig.text(
    0.01,
    -0.02,
    "Dutch day-ahead prices, October 2025 to August 2026. "
    "Source: ENTSO-E Transparency Platform; analysis/04_shifting_gain.sql",
    fontsize=8,
    color=MUTED,
)

Path("../figures").mkdir(exist_ok=True)
fig.savefig("../figures/savings_by_hours_moved.png", dpi=200, bbox_inches="tight")
plt.show()